# eICU-CRD 2.0: Cohort Construction

## 1 - Setup

In [ ]:
import os, glob, time, zipfile, numpy as np, pandas as pd
try:
    import duckdb
except ImportError:
    import subprocess; subprocess.run(["pip", "-q", "install", "duckdb"], check=True); import duckdb
print("duckdb", duckdb.__version__, "| pandas", pd.__version__)
from google.colab import drive
drive.mount('/content/drive')

## 2 - Configuration and unzip

In [ ]:
EICU_ZIP = "..."
EXTRACT  = "..."
OUT_NPZ  = "..."

T          = 48          # hours
WINDOW_MIN = T * 60      # 48 h in minutes
LOS_MIN    = 2 * 24 * 60 # ICU LOS >= 2 days
MIN_AGE    = 18

if not glob.glob(f"{EXTRACT}/**/patient.csv.gz", recursive=True):
    os.makedirs(EXTRACT, exist_ok=True)
    print("unzipping eICU (a few minutes)...")
    with zipfile.ZipFile(EICU_ZIP) as z:
        z.extractall(EXTRACT)
BASE = os.path.dirname(glob.glob(f"{EXTRACT}/**/patient.csv.gz", recursive=True)[0])
def tbl(name): return os.path.join(BASE, f"{name}.csv.gz")
print("eICU tables at:", BASE)
print("files:", sorted(os.path.basename(f) for f in glob.glob(f"{BASE}/*.csv.gz"))[:12])

## 3 - Variable taxonomy (protocol / acuity / triggered) and source mapping

In [ ]:
VITAL_MAP = {"heartrate": "HeartRate", "respiration": "RespRate", "sao2": "SpO2", "temperature": "Temperature"}
APERIODIC_MAP = {"noninvasivesystolic": "NBP_sys", "noninvasivediastolic": "NBP_dia", "noninvasivemean": "NBP_mean"}
GCS_MAP = {"Eyes": "GCS_eye", "Verbal": "GCS_verbal", "Motor": "GCS_motor"}

LAB_MAP = {
    "creatinine": "Creatinine", "BUN": "BUN", "sodium": "Sodium", "potassium": "Potassium",
    "chloride": "Chloride", "bicarbonate": "Bicarbonate", "glucose": "Glucose", "Hct": "Hematocrit",
    "Hgb": "Hemoglobin", "WBC x 1000": "WBC", "platelets x 1000": "Platelets", "magnesium": "Magnesium",
    "calcium": "Calcium", "phosphate": "Phosphate",
    "lactate": "Lactate", "troponin - T": "Troponin_T", "pH": "pH", "paO2": "pO2", "paCO2": "pCO2",
    "Base Excess": "BaseExcess", "total bilirubin": "Bilirubin", "PT - INR": "INR", "PT": "PT",
    "PTT": "PTT", "albumin": "Albumin",
}
PROTOCOL = list(VITAL_MAP.values()) + list(APERIODIC_MAP.values()) + list(GCS_MAP.values())
ACUITY   = ["Creatinine","BUN","Sodium","Potassium","Chloride","Bicarbonate","Glucose","Hematocrit",
            "Hemoglobin","WBC","Platelets","Magnesium","Calcium","Phosphate"]
TRIGGERED= ["Lactate","Troponin_T","pH","pO2","pCO2","BaseExcess","Bilirubin","INR","PT","PTT","Albumin"]
VAR_NAMES = PROTOCOL + ACUITY + TRIGGERED
VAR_CLASS = ["protocol"]*len(PROTOCOL) + ["acuity"]*len(ACUITY) + ["triggered"]*len(TRIGGERED)
vidx = {v: i for i, v in enumerate(VAR_NAMES)}; V = len(VAR_NAMES)
print(f"{V} variables | protocol {len(PROTOCOL)} acuity {len(ACUITY)} triggered {len(TRIGGERED)}")

## 4 - Discovery: verify labnames and GCS labels present

In [ ]:
con = duckdb.connect()
print("=== top 40 labnames by frequency (match these to LAB_MAP) ===")
top = con.sql(
    f"SELECT labname, count(*) AS n FROM read_csv_auto('{tbl('lab')}') "
    "GROUP BY labname ORDER BY n DESC LIMIT 40").df()
print(top.to_string(index=False))
missing = [k for k in LAB_MAP if k not in set(top.labname)]
print("\nLAB_MAP keys NOT in top-40 (check spelling / rarity):", missing)
print("\n=== GCS component labels in nurseCharting (expect Eyes/Verbal/Motor) ===")
gcs = con.sql(
    f"SELECT nursingchartcelltypevalname AS comp, count(*) AS n FROM read_csv_auto('{tbl('nurseCharting')}') "
    "WHERE nursingchartcelltypevallabel = 'Glasgow coma score' "
    "GROUP BY comp ORDER BY n DESC LIMIT 10").df()
print(gcs.to_string(index=False))

## 5 - Cohort selection and labels (from `patient`)

In [ ]:
pat = con.sql(
    f"SELECT patientunitstayid AS stay, age, gender, unittype, unitvisitnumber, "
    "unitdischargeoffset, hospitaldischargestatus, unitdischargestatus "
    f"FROM read_csv_auto('{tbl('patient')}')").df()
def parse_age(a):
    a = str(a).strip()
    if a in (">89", "> 89"): return 90.0
    try: return float(a)
    except Exception: return np.nan
pat["age_num"] = pat["age"].map(parse_age)
coh = pat[(pat.unitvisitnumber == 1) & (pat.age_num >= MIN_AGE) &
          (pat.unitdischargeoffset >= LOS_MIN)].dropna(subset=["age_num"]).copy()
coh = coh.drop_duplicates("stay").reset_index(drop=True)
coh["row"] = np.arange(len(coh))
stay2row = dict(zip(coh.stay, coh.row)); N = len(coh)

labels = {"mortality_inhosp": (coh.hospitaldischargestatus == "Expired").astype(int).values,
          "icu_mortality":    (coh.unitdischargestatus == "Expired").astype(int).values}
age_z = ((coh.age_num - coh.age_num.mean()) / (coh.age_num.std() + 1e-6)).values.astype(np.float32)
is_female = (coh.gender == "Female").astype(np.float32).values
unittype_oh = pd.get_dummies(coh.unittype, prefix="unit").astype(np.float32)
cov_names = ["age_z", "is_female"] + list(unittype_oh.columns)
c = np.column_stack([age_z, is_female, unittype_oh.values]).astype(np.float32)
print(f"cohort N={N} | d_c={c.shape[1]}")
for k, v in labels.items(): print(f"  {k:18s} prevalence {v.mean():.3f}")
con.register("cohort_stays", coh[["stay"]])

## 6 - Extract + hourly-bin each source 

In [ ]:
long_frames = []
def melt_wide(df, colmap):
    d = df.rename(columns=colmap)
    keep = ["stay", "hr"] + list(colmap.values())
    return d[keep].melt(id_vars=["stay", "hr"], var_name="var", value_name="val").dropna(subset=["val"])

t0 = time.time()
vp = con.sql(
    f"SELECT v.patientunitstayid AS stay, CAST(v.observationoffset/60 AS INTEGER) AS hr, "
    "avg(v.heartrate) heartrate, avg(v.respiration) respiration, avg(v.sao2) sao2, avg(v.temperature) temperature "
    f"FROM read_csv_auto('{tbl('vitalPeriodic')}') v "
    "JOIN cohort_stays c ON v.patientunitstayid = c.stay "
    f"WHERE v.observationoffset >= 0 AND v.observationoffset < {WINDOW_MIN} "
    "GROUP BY stay, hr").df()
long_frames.append(melt_wide(vp, VITAL_MAP)); print(f"vitalPeriodic {len(vp)} bins | {time.time()-t0:.0f}s")

t0 = time.time()
va = con.sql(
    f"SELECT v.patientunitstayid AS stay, CAST(v.observationoffset/60 AS INTEGER) AS hr, "
    "avg(v.noninvasivesystolic) noninvasivesystolic, avg(v.noninvasivediastolic) noninvasivediastolic, "
    "avg(v.noninvasivemean) noninvasivemean "
    f"FROM read_csv_auto('{tbl('vitalAperiodic')}') v "
    "JOIN cohort_stays c ON v.patientunitstayid = c.stay "
    f"WHERE v.observationoffset >= 0 AND v.observationoffset < {WINDOW_MIN} "
    "GROUP BY stay, hr").df()
long_frames.append(melt_wide(va, APERIODIC_MAP)); print(f"vitalAperiodic {len(va)} bins | {time.time()-t0:.0f}s")

t0 = time.time()
nc = con.sql(
    f"SELECT n.patientunitstayid AS stay, CAST(n.nursingchartoffset/60 AS INTEGER) AS hr, "
    "n.nursingchartcelltypevalname AS comp, avg(TRY_CAST(n.nursingchartvalue AS DOUBLE)) AS val "
    f"FROM read_csv_auto('{tbl('nurseCharting')}') n "
    "JOIN cohort_stays c ON n.patientunitstayid = c.stay "
    f"WHERE n.nursingchartoffset >= 0 AND n.nursingchartoffset < {WINDOW_MIN} "
    "AND n.nursingchartcelltypevallabel = 'Glasgow coma score' "
    "AND n.nursingchartcelltypevalname IN ('Eyes','Verbal','Motor') "
    "GROUP BY stay, hr, comp").df()
nc["var"] = nc["comp"].map(GCS_MAP); nc = nc.dropna(subset=["var", "val"])
long_frames.append(nc[["stay", "hr", "var", "val"]]); print(f"nurseCharting/GCS {len(nc)} bins | {time.time()-t0:.0f}s")

t0 = time.time()
lab_keys = tuple(LAB_MAP.keys())
lb = con.sql(
    f"SELECT l.patientunitstayid AS stay, CAST(l.labresultoffset/60 AS INTEGER) AS hr, "
    "l.labname AS labname, avg(l.labresult) AS val "
    f"FROM read_csv_auto('{tbl('lab')}') l "
    "JOIN cohort_stays c ON l.patientunitstayid = c.stay "
    f"WHERE l.labresultoffset >= 0 AND l.labresultoffset < {WINDOW_MIN} AND l.labname IN {lab_keys} "
    "GROUP BY stay, hr, labname").df()
lb["var"] = lb["labname"].map(LAB_MAP); lb = lb.dropna(subset=["var", "val"])
long_frames.append(lb[["stay", "hr", "var", "val"]]); print(f"lab {len(lb)} bins | {time.time()-t0:.0f}s")

## 7 - Assemble the (N, T, V) mask/value arrays

In [ ]:
long_all = pd.concat(long_frames, ignore_index=True)
m = np.zeros((N, T, V), np.float32); y = np.zeros((N, T, V), np.float32)
rows = long_all["stay"].map(stay2row).to_numpy()
cols = long_all["var"].map(vidx).to_numpy()
hrs  = long_all["hr"].to_numpy()
vals = long_all["val"].to_numpy()
ok = (~pd.isna(rows)) & (~pd.isna(cols)) & (hrs >= 0) & (hrs < T) & (~pd.isna(vals))
r = rows[ok].astype(int); h = hrs[ok].astype(int); cc = cols[ok].astype(int); vv = vals[ok].astype(np.float32)
y[r, h, cc] = vv; m[r, h, cc] = 1.0
print(f"assembled arrays: m,y {m.shape} | placed {ok.sum()} observations | overall mask density {m.mean():.4f}")
dens = m.mean((0, 1))
print("\nper-variable density (name: density | class):")
for i, nm in enumerate(VAR_NAMES):
    print(f"  {nm:14s} {dens[i]:.3f}  {VAR_CLASS[i]}")

## 8 - Save the cohort (.npz) - identical structure to the MIMIC cache

In [ ]:
np.savez_compressed(
    OUT_NPZ,
    m=m.astype(np.float32), y=y.astype(np.float32), c=c.astype(np.float32),
    cov_names=np.array(cov_names, dtype=object),
    var_names=np.array(VAR_NAMES, dtype=object), var_class=np.array(VAR_CLASS, dtype=object),
    stay_id=coh.stay.values.astype(np.int64),
    **{f"lab_{k}": v.astype(np.int64) for k, v in labels.items()})
print("saved eICU cohort ->", OUT_NPZ)
d = np.load(OUT_NPZ, allow_pickle=True)
print("keys:", list(d.files))
print(f"shapes: m {d['m'].shape} y {d['y'].shape} c {d['c'].shape} | V={len(d['var_names'])}")